In [3]:
%pip install duckdb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 8.5 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [10]:
import duckdb
import pandas as pd

RAW = "/users/qpingwin/documents/claude_all/uk-housing-analysis/data/raw/pp-complete.csv"

# Column names — the file has NO header row
COLS = [
    "transaction_id", "price", "transfer_date", "postcode",
    "property_type", "old_new", "duration", "paon", "saon",
    "street", "locality", "town", "district", "county",
    "ppd_category", "record_status"
]

con = duckdb.connect()

# Tell DuckDB the file has no header and give it column names
con.execute(f"""
    CREATE VIEW ppd AS
    SELECT * FROM read_csv(
        '{RAW}',
        header=false,
        columns={{
            'transaction_id': 'VARCHAR',
            'price':          'BIGINT',
            'transfer_date':  'VARCHAR',   -- parse manually later
            'postcode':       'VARCHAR',
            'property_type':  'VARCHAR',
            'old_new':        'VARCHAR',
            'duration':       'VARCHAR',
            'paon':           'VARCHAR',
            'saon':           'VARCHAR',
            'street':         'VARCHAR',
            'locality':       'VARCHAR',
            'town':           'VARCHAR',
            'district':       'VARCHAR',
            'county':         'VARCHAR',
            'ppd_category':   'VARCHAR',
            'record_status':  'VARCHAR'
        }}
    )
""")

# --- 1. Total row count and Category A/B split
print("=== Row count by category ===")
print(con.execute("""
    SELECT ppd_category, COUNT(*) AS n_rows
    FROM ppd
    GROUP BY ppd_category
    ORDER BY ppd_category
""").df())

# --- 2. Property type distribution
print("\n=== Property type ===")
print(con.execute("""
    SELECT property_type, COUNT(*) AS n, ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1) AS pct
    FROM ppd
    WHERE ppd_category = 'A'
    GROUP BY property_type
    ORDER BY n DESC
""").df())

# --- 3. New build vs existing
print("\n=== New build ===")
print(con.execute("""
    SELECT old_new, COUNT(*) AS n
    FROM ppd WHERE ppd_category = 'A'
    GROUP BY old_new
""").df())

# --- 4. Missing postcodes
print("\n=== Missing postcodes ===")
print(con.execute("""
    SELECT
        COUNT(*) FILTER (WHERE postcode IS NULL OR TRIM(postcode) = '') AS null_postcode,
        COUNT(*) AS total
    FROM ppd WHERE ppd_category = 'A'
""").df())

# --- 5. Year distribution — check coverage is complete
print("\n=== Transactions per year (Cat A) ===")
print(con.execute("""
    SELECT
        CAST(transfer_date[:4] AS INTEGER) AS year,
        COUNT(*) AS n_sales
    FROM ppd
    WHERE ppd_category = 'A'
    GROUP BY year
    ORDER BY year
""").df().to_string())

# --- 6. Price sanity check
print("\n=== Price distribution (Cat A) ===")
print(con.execute("""
    SELECT
        MIN(price)    AS min_price,
        MAX(price)    AS max_price,
        MEDIAN(price) AS median_price,
        COUNT(*) FILTER (WHERE price < 10000) AS suspiciously_low
    FROM ppd WHERE ppd_category = 'A'
""").df())

# --- 7. Duplicate transaction IDs
print("\n=== Duplicate transaction IDs ===")
print(con.execute("""
    SELECT COUNT(*) - COUNT(DISTINCT transaction_id) AS duplicates
    FROM ppd
""").df())

=== Row count by category ===
  ppd_category    n_rows
0            A  29490159
1            B   1780116

=== Property type ===
  property_type        n   pct
0             T  8804631  29.9
1             S  8298959  28.1
2             D  7089072  24.0
3             F  5297497  18.0

=== New build ===
  old_new         n
0       Y   3097448
1       N  26392711

=== Missing postcodes ===
   null_postcode     total
0          14203  29490159

=== Transactions per year (Cat A) ===
    year  n_sales
0   1995   797122
1   1996   965391
2   1997  1094612
3   1998  1050656
4   1999  1195030
5   2000  1129513
6   2001  1245976
7   2002  1351910
8   2003  1235580
9   2004  1232084
10  2005  1061700
11  2006  1326138
12  2007  1272315
13  2008   649525
14  2009   625203
15  2010   663154
16  2011   660972
17  2012   668535
18  2013   791644
19  2014   919348
20  2015   918868
21  2016   922384
22  2017   906869
23  2018   875543
24  2019   842986
25  2020   752934
26  2021  1086573
27  2022   909

In [8]:
con.execute(f"SELECT * FROM read_csv_auto('{RAW}') LIMIT 10").df()

,column00,column01,column02,column03,column04,column05,column06,column07,column08,column09,column10,column11,column12,column13,column14,column15
0,{2A289E9F-6BB5-CDC8-E050-A8C063054829},36995,1995-03-24,SE19 3NF,F,N,L,CROWN POINT,14,BEULAH HILL,None,LONDON,CROYDON,GREATER LONDON,A,A
1,{2A289E9F-6BBA-CDC8-E050-A8C063054829},25000,1995-03-31,E16 1LG,F,N,L,9,NaN,POLLARD CLOSE,None,LONDON,NEWHAM,GREATER LONDON,A,A
2,{2A289E9F-6BC5-CDC8-E050-A8C063054829},25500,1995-05-17,EN3 6EA,F,N,L,33,NaN,BRIDLE CLOSE,None,ENFIELD,ENFIELD,GREATER LONDON,A,A
3,{2A289E9F-7DE9-CDC8-E050-A8C063054829},42000,1995-04-21,N13 4RS,T,N,L,87,NaN,RUSSELL ROAD,None,LONDON,ENFIELD,GREATER LONDON,A,A
4,{2A289E9F-7DF0-CDC8-E050-A8C063054829},43000,1995-06-30,RM10 7NU,T,N,F,45,NaN,BOSWORTH ROAD,None,DAGENHAM,BARKING AND DAGENHAM,GREATER LONDON,A,A
5,{2A289E9F-7DF5-CDC8-E050-A8C063054829},6500,1995-07-21,BB10 1BY,T,N,L,10,NaN,HURTLEY STREET,None,BURNLEY,BURNLEY,LANCASHIRE,A,A
6,{2A289E9F-7DFF-CDC8-E050-A8C063054829},38000,1995-06-07,CR0 6NB,F,N,L,54,THE GROUND FLOOR FLAT AT,MORLAND ROAD,None,CROYDON,CROYDON,GREATER LONDON,A,A
7,{2A289E9F-7E0D-CDC8-E050-A8C063054829},55000,1995-06-06,BD8 8RA,S,N,F,32,NaN,HEATON ROAD,None,BRADFORD,BRADFORD,WEST YORKSHIRE,A,A
8,{2A289E9F-7E10-CDC8-E050-A8C063054829},38000,1995-06-06,SW16 5LL,F,N,L,25,FLAT 2,TANKERVILLE ROAD,None,LONDON,LAMBETH,GREATER LONDON,A,A
9,{2A289E9F-7E1A-CDC8-E050-A8C063054829},49950,1995-04-28,SW19 2HB,F,N,L,14A,NaN,WILTON ROAD,None,LONDON,MERTON,GREATER LONDON,A,A


In [9]:
onspd_cols = ["pcds", "lad25cd", "rgn25cd", "ctry25cd", "lat", "long"]
onspd = pd.read_csv("/users/qpingwin/documents/claude_all/uk-housing-analysis/data/raw/ONSPD_MAY_2026_UK.csv", usecols=onspd_cols, low_memory=False)